# 07 — Rectified Flow retrained on the training split ONLY

**Blocking fix.** The existing checkpoint was trained by `rectified_flow/train.py`, which calls
`sklearn.train_test_split` over *every* image. The generator therefore saw the classifier's test
partition, and any augmentation result derived from it is contaminated.

This notebook fixes the data, not the model: same architecture, same objective, same
hyperparameters — the generator only ever sees images whose blocks are in the classifier's
training partition.

Upload `main_thesis!/some_extra_Experiment/rectified_flow/` as a Kaggle dataset named
`rf-code`. Budget ~6 GPU-h on 2xT4 -- **or run it on TPU v5e instead** using your existing
`some_extra_Experiment/tpu/rectified_flow_jax.py`, which hit 576 img/s. Doing so keeps the
entire GPU quota for the classifier experiments, which is the right allocation.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 1. Freeze the canonical split that ALL downstream experiments will share.
CANON_SEED, CANON_STRAT = 42, "block_shuffled"
recs = splits.make_split(splits.build_registry(ROOT), CANON_STRAT, seed=CANON_SEED)
rep = splits.verify_split(recs)
assert rep["blocks_are_disjoint"], rep
print(splits.split_table(recs))

train_paths = [r["image_path"] for r in recs if r["split"] == "train"]
train_labels = [r["label"] for r in recs if r["split"] == "train"]
holdout = {r["image_path"] for r in recs if r["split"] in ("val", "test")}
assert not (set(train_paths) & holdout), "train/holdout overlap"

manifest = {"strategy": CANON_STRAT, "seed": CANON_SEED,
            "train_paths": train_paths, "train_labels": train_labels,
            "n_train": len(train_paths), "n_holdout": len(holdout)}
os.makedirs(f"{RESULTS}/07_rf", exist_ok=True)
with open(f"{RESULTS}/07_rf/generator_train_manifest.json", "w") as f:
    json.dump(manifest, f)
print(f"\ngenerator may see {len(train_paths)} images; {len(holdout)} are off limits")

In [ ]:
# 2. Patch the existing trainer so it reads the manifest instead of splitting randomly.
import shutil, os
SRC = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train.py" in files and "unet.py" in files and not SRC:
            SRC = root
if not SRC and os.path.isdir("/kaggle/input/rf-code/rectified_flow"):
    SRC = "/kaggle/input/rf-code/rectified_flow"
elif not SRC and os.path.isdir("/kaggle/input/rf-code"):
    SRC = "/kaggle/input/rf-code"
assert SRC and os.path.isdir(SRC), f"rf-code source not found in /kaggle/input; tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"
print("Found rf-code source at:", SRC)

DST = "/kaggle/working/rectified_flow"
if os.path.isdir(DST): shutil.rmtree(DST)
shutil.copytree(SRC, DST)

p = f"{DST}/train.py"
src = open(p).read()

patch = f"""# ---- PATCHED: generator sees the classifier TRAIN partition only ----
import json as _json
_mf = _json.load(open('{RESULTS}/07_rf/generator_train_manifest.json'))
train_paths, train_labels = _mf['train_paths'], _mf['train_labels']
_n = max(1, int(0.1 * len(train_paths)))
val_paths, val_labels = train_paths[:_n], train_labels[:_n]
test_paths, test_labels = [], []
"""

idx1 = src.find("train_paths, temp_paths")
idx2 = src.find("if is_main_process():", idx1)
assert idx1 != -1 and idx2 != -1, "could not locate the train_test_split block -- inspect train.py by hand"
src = src[:idx1] + patch + src[idx2:]
open(p, "w").write(src)

assert "train_test_split(" not in src.split("PATCHED")[1][:2000], "a random split survived"
print("patched. verifying no holdout image can reach the generator...")
import json as _j
_mf = _j.load(open(f"{RESULTS}/07_rf/generator_train_manifest.json"))
assert not (set(_mf["train_paths"]) & holdout)
print("OK -- generator training set is clean")


In [ ]:
# 3. Train on 2x T4 GPUs using torchrun.
import os
os.chdir("/kaggle/working")
!torchrun --nproc_per_node=2 rectified_flow/train.py


In [ ]:
# 4. Generate synthetic Benign images for the augmentation study.
import os
ckpt = "/kaggle/working/rf_output/best_rf_model.pt" if os.path.exists("/kaggle/working/rf_output/best_rf_model.pt") else "/kaggle/working/best_rf_model.pt"
!python rectified_flow/generate.py --checkpoint "" \n        --output_dir /kaggle/working/synthetic_benign --num_images 300 --class_name Benign
print(len(os.listdir("/kaggle/working/synthetic_benign")))
